In [76]:
import numpy as np
import pandas as pd

In [77]:
from random import randint
from scipy.spatial.transform import Rotation as R

def rotate_thf(thf):
    """
    This function randomizes the orientation of THF in space. It does this with a transform matrix
    provided by the scipy Rotation package. By using this package, THF can be rotated around it's
    center in 3D space, and not just around an axis. This funtion also provides random rotations,
    with a random number of degrees in each direction.

    For more information on the theory, visit:
    https://stackoverflow.com/questions/14607640/rotating-a-vector-in-3d-space

    Input: THF dataframe that includes columns for 'X', 'Y', and 'Z'

    Output: Identical THF dataframe with updated coordinates given a random rotation in 3D space

    Author: Gabe Miles
    """

    x,y,z = randint(0,360), randint(0,360), randint(0,360)
    r = R.from_euler('zyx', [x,y,z], degrees=True)
    tmp = np.empty((0,3), int)
    for i in range(len(thf.index)):
        # np.vstack returns the column vector of 'zyx', and the '@' does matrix multiplication
        new_xyz = r.as_matrix() @ np.vstack(thf[['X','Y','Z']].iloc[i].values)
        tmp = np.append(tmp, new_xyz.T, axis=0)

    rot_thf = pd.DataFrame({'ELEMENT': thf['ELEMENT'],
                        'X': tmp[:,[0]].T[0],
                        'Y': tmp[:,[1]].T[0],
                        'Z': tmp[:,[2]].T[0],
                        'RESIDUE_NUM': thf['RESIDUE_NUM'],
                        'MOL': thf['MOL']})
    
    return rot_thf

In [78]:
pdb=pd.read_csv('output/low_hydrate.pdb', header=0, names=['TYPE','ID','ELEMENT','MOL','CHAIN_ID','RESIDUE_NUM','X','Y','Z','NUM1','NUM2'], delim_whitespace=True)
rando_hydrate = pdb.drop(columns=['TYPE','ID','CHAIN_ID','NUM1','NUM2'])
cages=pd.read_csv('input/s2-cages.csv')
thf=pd.read_csv('input/thf.csv')

In [79]:
index = len(rando_hydrate.index)
rando_hydrate.loc[3997:index, ['X','Y','Z']] = pdb.loc[3997:index, ['RESIDUE_NUM','X','Y']].values

# Hydrate has incorrect residue numbers, and they are floats
rando_hydrate = rando_hydrate.drop(['RESIDUE_NUM'], axis=1)
rando_hydrate['RESIDUE_NUM'] = 0
residue_num = 1
for i in range(0,len(rando_hydrate.index)):
    rando_hydrate.loc[i,'RESIDUE_NUM'] = residue_num
    if rando_hydrate.loc[i,'ELEMENT'] == 'MW':
        residue_num += 1
# = pdb[['RESIDUE_NUM','X','Y']].iloc[3997:len(rando_hydrate.index)].values

In [80]:
# Insert THF
large_cages = cages[cages['TYPE'].str.contains('Large')]
thf_center = thf[['X', 'Y', 'Z']].mean()
# Set center to (0,0,0)
thf[['X','Y','Z']] = thf[['X','Y','Z']] - thf_center
# Update to new center
thf['RESIDUE_NUM'] = -1
thf['MOL'] = 'THF'

In [81]:
#Make the box 3 x 3
cell_size = 17.30
num_cells = 3
coord_offset = [[-cell_size, 0, cell_size],[0,0,cell_size],[cell_size, 0, cell_size],
                [-cell_size, 0, 0],[cell_size, 0, 0],
                [-cell_size, 0, -cell_size],[0, 0, -cell_size],[cell_size, 0, -cell_size]]

In [82]:
cages_3_plane = large_cages.copy()
for offset in coord_offset:
    # For hydrate
    large_cages_unit = large_cages.copy()
    large_cages_unit[['X','Y','Z']] = large_cages_unit[['X','Y','Z']] + offset
    cages_3_plane = pd.concat([cages_3_plane, large_cages_unit], axis=0, ignore_index=True)

In [83]:
# Stack 3x3x3 crystal
cages_3_3 = cages_3_plane.copy()
for i in range(1, num_cells):
    cages_unit = cages_3_plane.copy()
    cages_unit['Y'] = cages_unit['Y'] + (cell_size * i)
    cages_3_3 = pd.concat([cages_3_3, cages_unit], axis=0, ignore_index=True)

In [84]:
rando_hydrate.info

<bound method DataFrame.info of       ELEMENT  MOL       X       Y       Z  RESIDUE_NUM
0          OW  HOH   5.925   5.925   5.925            1
1         HW1  HOH   6.866   5.919   5.749            1
2         HW2  HOH   5.823   6.525   6.664            1
3          MW  HOH   6.032   6.001   5.997            1
4          OW  HOH   5.322   5.322   8.566            2
...       ...  ...     ...     ...     ...          ...
14679      MW  HOH  30.404  47.675 -12.950         3670
14680      OW  HOH  21.627  47.593  -4.307         3671
14681     HW1  HOH  20.799  47.156  -4.111         3671
14682     HW2  HOH  21.557  48.445  -3.875         3671
14683      MW  HOH  21.512  47.646  -4.226         3671

[14684 rows x 6 columns]>

In [85]:
# This loop recenters THF and places it in all 8 large cages of the sII hydrate
thf_hydrate_randomized_3_3 = rando_hydrate.copy()
for i in range(len(cages_3_3.index)):
    rot_thf = rotate_thf(thf)
    dxyz = thf_center - cages_3_3[['X','Y','Z']].iloc[i]
    thf_translated = rot_thf.copy()
    thf_translated[['X','Y','Z']] = thf_translated[['X','Y','Z']] - dxyz
    thf_translated['RESIDUE_NUM'] = thf_hydrate_randomized_3_3['RESIDUE_NUM'].iloc[len(thf_hydrate_randomized_3_3.index) - 1] + 1
    thf_hydrate_randomized_3_3 = pd.concat([thf_hydrate_randomized_3_3, thf_translated], ignore_index=True)
    residue_num += 1

thf_hydrate_randomized_3_3

,ELEMENT,MOL,X,Y,Z,RESIDUE_NUM
0,OW,HOH,5.925000,5.925000,5.925000,1
1,HW1,HOH,6.866000,5.919000,5.749000,1
2,HW2,HOH,5.823000,6.525000,6.664000,1
3,MW,HOH,6.032000,6.001000,5.997000,1
4,OW,HOH,5.322000,5.322000,8.566000,2
...,...,...,...,...,...,...
17487,H08,THF,15.953609,43.615077,-17.604836,3887
17488,H09,THF,18.203427,41.581205,-17.446565,3887
17489,H0A,THF,16.763698,41.664909,-16.413081,3887
17490,H0B,THF,19.114388,42.815413,-15.624024,3887


In [87]:
f = open("output/tip4p_thf_hydrate_randomized_0_dipole.pdb", "w")

f.write("CRYST1   52.000   52.000   52.000  90.00  90.00  90.00 P 1           1\n")
for i in range(len(thf_hydrate_randomized_3_3.index)):
    #Collect PDB variables
    id = str(i)
    chainid = 'A'
    atom = thf_hydrate_randomized_3_3['ELEMENT'].iloc[i]
    resname = thf_hydrate_randomized_3_3['MOL'].iloc[i]
    resnum = thf_hydrate_randomized_3_3['RESIDUE_NUM'].iloc[i]
    x = thf_hydrate_randomized_3_3['X'].iloc[i]
    y = thf_hydrate_randomized_3_3['Y'].iloc[i]
    z = thf_hydrate_randomized_3_3['Z'].iloc[i]

    # Create PDB entry
    line = "{:6s}{:5d} {:^4s} {:3s} {:1s}{:5d}    {:8.3f}{:8.3f}{:8.3f}{:6.2f}{:6.2f}\n".format(
           "ATOM", i+1, atom, resname, chainid, resnum, x, y, z, 0., 0.)
    f.write(line)

f.close()